# FraudShield — Exploratory Data Analysis

Dataset: [Kaggle Credit Card Fraud Detection (ULB)](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
284,807 transactions · 492 frauds · V1–V28 PCA features + Time + Amount

**Questions:**
1. How imbalanced is the dataset?
2. How do fraud and legitimate transactions differ in amount?
3. Are there temporal patterns?
4. Any missing values?
5. Which features correlate most with fraud?

In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for nbconvert
import matplotlib.pyplot as plt
import seaborn as sns

from ml.config import RAW_DATA_PATH, ALL_FEATURES, TARGET, DOCS_DIR

sns.set_theme(style='whitegrid', palette='husl')
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete ✓')

Setup complete ✓


In [2]:
df = pd.read_csv(RAW_DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

Shape: (284807, 31)
Columns: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## 1. Class Balance

The dataset is extremely imbalanced — fraud is about 0.17 % of transactions.

In [3]:
fraud_counts = df[TARGET].value_counts().sort_index()
print(f'Legitimate (0): {fraud_counts[0]:>7,}  '
      f'({fraud_counts[0] / len(df) * 100:.3f}%)')
print(f'Fraud      (1): {fraud_counts[1]:>7,}  '
      f'({fraud_counts[1] / len(df) * 100:.3f}%)')
print(f'Imbalance ratio: 1 : {fraud_counts[0] // fraud_counts[1]}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#2ecc71', '#e74c3c']
bars = axes[0].bar(['Legitimate', 'Fraud'],
                   fraud_counts.values, color=colors)
axes[0].set_ylabel('Count')
axes[0].set_title('Class Distribution (absolute)')
for bar, cnt in zip(bars, fraud_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                bar.get_height(), f'{cnt:,}',
                ha='center', va='bottom', fontweight='bold')

axes[1].pie(fraud_counts.values,
            labels=['Legitimate', 'Fraud'],
            colors=colors, autopct='%1.3f%%', startangle=90)
axes[1].set_title('Class Distribution (proportion)')

plt.tight_layout()
plt.savefig(DOCS_DIR / 'class_distribution.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved → docs/class_distribution.png')

Legitimate (0): 284,315  (99.827%)
Fraud      (1):     492  (0.173%)
Imbalance ratio: 1 : 577
Saved → docs/class_distribution.png


C:\Users\Acer\AppData\Local\Temp\ipykernel_9516\1189667604.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Amount Distribution by Class

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[df[TARGET] == 0]['Amount'], bins=100,
             color='#2ecc71', alpha=0.7, log=True)
axes[0].set_title('Amount — Legitimate')
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Count (log scale)')

axes[1].hist(df[df[TARGET] == 1]['Amount'], bins=50,
             color='#e74c3c', alpha=0.7)
axes[1].set_title('Amount — Fraud')
axes[1].set_xlabel('Amount')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(DOCS_DIR / 'amount_distribution.png',
            dpi=150, bbox_inches='tight')
plt.show()

print('\nAmount statistics by class:')
print(df.groupby(TARGET)['Amount'].describe().round(2))


Amount statistics by class:
          count    mean     std  min   25%    50%     75%       max
Class                                                              
0      284315.0   88.29  250.11  0.0  5.65  22.00   77.05  25691.16
1         492.0  122.21  256.68  0.0  1.00   9.25  105.89   2125.87


C:\Users\Acer\AppData\Local\Temp\ipykernel_9516\3266196326.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Time Patterns

`Time` = seconds elapsed from the first transaction in the dataset (~48 hours).

In [5]:
hours = df['Time'] / 3600

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].hist(hours, bins=100, color='#3498db', alpha=0.7)
axes[0].set_title('All Transactions Over Time')
axes[0].set_ylabel('Count')

axes[1].hist(hours[df[TARGET] == 1], bins=48,
             color='#e74c3c', alpha=0.7)
axes[1].set_title('Fraud Transactions Over Time')
axes[1].set_xlabel('Time (hours)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(DOCS_DIR / 'time_patterns.png',
            dpi=150, bbox_inches='tight')
plt.show()

C:\Users\Acer\AppData\Local\Temp\ipykernel_9516\2289321890.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Missing Values

In [6]:
missing = df.isnull().sum()
total_missing = missing.sum()
print(f'Total missing values across all columns: {total_missing}')

if total_missing > 0:
    print('\nColumns with missing values:')
    print(missing[missing > 0])
else:
    print('No missing values found ✓')

Total missing values across all columns: 0
No missing values found ✓


## 5. Feature Correlations with Fraud

In [7]:
corr_with_target = (
    df[ALL_FEATURES + [TARGET]]
    .corr()[TARGET]
    .drop(TARGET)
    .sort_values()
)

fig, ax = plt.subplots(figsize=(10, 12))
colors = ['#e74c3c' if v < 0 else '#2ecc71'
          for v in corr_with_target.values]
corr_with_target.plot.barh(ax=ax, color=colors)
ax.set_title('Feature Correlation with Fraud (Pearson)')
ax.set_xlabel('Correlation coefficient')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig(DOCS_DIR / 'feature_correlations.png',
            dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 features most correlated with fraud (abs):')
print(corr_with_target.abs().sort_values(ascending=False).head(10).round(4))


Top 10 features most correlated with fraud (abs):
V17    0.3265
V14    0.3025
V12    0.2606
V10    0.2169
V16    0.1965
V3     0.1930
V7     0.1873
V11    0.1549
V4     0.1334
V18    0.1115
Name: Class, dtype: float64


C:\Users\Acer\AppData\Local\Temp\ipykernel_9516\4207066219.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. V-Feature Correlation Heatmap

V1–V28 are PCA-transformed, so they should be largely uncorrelated.

In [8]:
fig, ax = plt.subplots(figsize=(14, 12))
pca_cols = [f'V{i}' for i in range(1, 29)]
corr_matrix = df[pca_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='RdBu_r',
            center=0, square=True, linewidths=0.5,
            ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('V-Feature Pairwise Correlations')
plt.tight_layout()
plt.savefig(DOCS_DIR / 'v_feature_heatmap.png',
            dpi=150, bbox_inches='tight')
plt.show()

C:\Users\Acer\AppData\Local\Temp\ipykernel_9516\402782266.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

| Finding | Detail |
|---|---|
| Class imbalance | ~0.17 % fraud — extreme imbalance |
| Amount | Fraud amounts tend lower; legitimate has a long tail |
| Time | Two visible daily cycles; fraud spread throughout |
| Missing values | None |
| Key features | V14, V17, V12, V10 most correlated with fraud |
| PCA features | Largely uncorrelated as expected |

**Next step →** Phase 2: baseline models with proper class-imbalance handling.